In [16]:
import sentencepiece as spm
from sentencepiece import sentencepiece_model_pb2 as sp_pb2_model

## Sarkaz Model

In [ ]:
import sentencepiece as spm
from sentencepiece import sentencepiece_model_pb2 as sp_pb2_model

In [21]:
def merge(original_model_path: str, domain_model_path: str, merged_model_path: str):
    # 1. 加载原始模型（二进制格式）
    print("Load original model ...")
    original_model = sp_pb2_model.ModelProto()
    with open(original_model_path, "rb") as f:
        original_model.ParseFromString(f.read())

    # 2. 加载领域模型（通过SentencePieceProcessor，再序列化为ModelProto）
    print("Load domain model ...")
    domain_processor = spm.SentencePieceProcessor()
    domain_processor.Load(domain_model_path)
    domain_model = sp_pb2_model.ModelProto()
    domain_model.ParseFromString(domain_processor.serialized_model_proto())

    # 3. 获取原始模型中已有的token集合（用于去重）
    original_tokens_set = set(p.piece for p in original_model.pieces)
    print(f"Original vocab size: {len(original_model.pieces)}")

    # 4. 合并：将领域模型中新出现的token添加进去
    new_token_count = 0
    for piece in domain_model.pieces:
        if piece.piece not in original_tokens_set:
            # 创建一个新的SentencePiece对象，并添加到原始模型中
            new_piece = original_model.pieces.add()
            new_piece.piece = piece.piece
            new_piece.score = 0.0   # 新token的分数可以设为0（训练时会自动调整）
            new_token_count += 1

    print(f"Add {new_token_count} new tokens")
    print(f"Vocab size (merged): {len(original_model.pieces)}")

    # 5. 保存合并后的模型文件
    with open(merged_model_path, "wb") as f:
        f.write(original_model.SerializeToString())
    print(f"Saved to: {merged_model_path}")

In [22]:
merge("models/sp_skz.model", "models/sp_ef_skz.model", "models/sp_merged_skz.model")
merge("models/sp_zh.model", "models/sp_ef_zh.model", "models/sp_merged_zh.model")

Load original model ...
Load domain model ...
Original vocab size: 512
Add 172 new tokens
Vocab size (merged): 684
Saved to: models/sp_merged_skz.model
Load original model ...
Load domain model ...
Original vocab size: 10000
Add 12417 new tokens
Vocab size (merged): 22417
Saved to: models/sp_merged_zh.model
